# Train on all public wound data: 7 wound types incl. burns, cross-validated

Adds seven public Kaggle wound datasets to the original AZH/FUSeg data, **deduplicated** (copies across datasets and of
the locked test photos are dropped) and mapped to **burn, pressure, diabetic, venous, surgical, other, no wound**:
about **4,820** photos for wound type and **2,582** traced photos for the outline. Then 5-fold cross-validation (3 if
5 won't fit), outline SegFormer-B2 at **768 px**.

**Before you start**
1. **Add Input** (right panel) → search and add each of these 7 datasets:
   `ibrahimfateen/wound-classification`, `yasinpratomo/wound-dataset`, `sinemgokoz/pressure-ulcers-stages`,
   `laithjj/diabetic-foot-ulcer-dfu`, `workshops11/medetec-dataset`, `shubhambaid/skin-burn-dataset`,
   `leoscode/wound-segmentation-images`.
2. Session options: Accelerator **GPU T4 x2**, Internet **on**, Persistence **Files only**.
3. **Save Version → Save & Run All (Commit)**.

The data step stops with a clear message if an input is missing. Research/learning use only: most of these datasets
state no licence.

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d wound_1 ] || git clone --depth 1 https://github.com/opalvedansh/wound_1.git
cd wound_1/wound-ai
pip install -q -r requirements.txt   # PyTorch is already installed on Kaggle
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
mkdir -p data/raw && cd data/raw
if [ ! -d fuseg ]; then
  git clone -q --depth 1 --filter=blob:none --sparse https://github.com/uwm-bigdata/wound-segmentation.git src
  (cd src && git sparse-checkout set --no-cone "/data/Foot Ulcer Segmentation Challenge/train/" "/data/Foot Ulcer Segmentation Challenge/validation/")
  mv "src/data/Foot Ulcer Segmentation Challenge" fuseg && rm -rf src
fi
if [ ! -d azh ]; then
  mkdir azh && cd azh
  for f in Train Test; do
    curl -sL -o $f.zip https://raw.githubusercontent.com/uwm-bigdata/wound-classification-using-images-and-locations/main/dataset/$f.zip
    unzip -q $f.zip && rm $f.zip
  done
  rm -rf __MACOSX && cd ..
fi
ls fuseg/train/images | wc -l; ls azh/Train

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
# Same seed and sources as on the Mac, so the splits match and test results are comparable.
python scripts/prepare_data.py \
  --pairs "fuseg:data/raw/fuseg/train/images:data/raw/fuseg/train/labels" \
          "fuseg:data/raw/fuseg/validation/images:data/raw/fuseg/validation/labels" \
  --out data/manifest.csv
python scripts/prepare_data.py \
  --classes "azh:data/raw/azh/Train:wound_type:configs/azh_map.json" "azh:data/raw/azh/Test:wound_type:configs/azh_map.json" \
  --out data/manifest_cls.csv

## 2. Add the public datasets (deduplicated, same result every run)

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
python scripts/link_kaggle_inputs.py
python scripts/build_public_dataset.py
python scripts/prepare_data.py --csv data/manifest_cls.csv data/public_cls.csv --out data/manifest_cls_v2.csv
python scripts/prepare_data.py --csv data/manifest.csv data/public_seg.csv --out data/manifest_seg_v2.csv
python - <<'EOF'
import pandas as pd
c, s = pd.read_csv('data/manifest_cls_v2.csv'), pd.read_csv('data/manifest_seg_v2.csv')
print(pd.crosstab(c.wound_type, c.split, margins=True))
print(f'wound type: {len(c)} photos (expected 4820) | outline: {len(s)} photos (expected 2582)')
if (len(c), len(s)) != (4820, 2582):
    print('NOTE: counts differ from the local build (a source dataset may have been updated); training continues.')
EOF

## 3. Cross-validation (several hours)

Times a short trial first and prints how many folds it will use, then one line per epoch per fold. With 2.5× the outline
photos, 40 epochs (patience 8) is enough.

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/cv.py --seg-manifest data/manifest_seg_v2.csv --cls-manifest data/manifest_cls_v2.csv \
  --folds 5 --fallback-folds 3 --budget-hours 10 \
  --seg-size 768 --seg-batch 4 --seg-epochs 40 --seg-patience 8 \
  --cls-size 384 --cls-batch 32 --cls-epochs 40 \
  --out /kaggle/working/cv

## 3. Package the results

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/outputs
cd /kaggle/working/cv
cp summary.md summary.json plan.json /kaggle/working/outputs/
cp -r best /kaggle/working/outputs/
for kind in seg cls; do
  for d in $kind/fold*; do
    f=$(basename $d)
    cp $d/eval_test.json /kaggle/working/outputs/${kind}_${f}_test.json 2>/dev/null || true
    cp $d/history.csv /kaggle/working/outputs/${kind}_${f}_history.csv 2>/dev/null || true
    cp $d/val_report.json /kaggle/working/outputs/${kind}_${f}_val.json 2>/dev/null || true
  done
done
cat /kaggle/working/outputs/summary.md
ls -la /kaggle/working/outputs /kaggle/working/outputs/best